# CS781 Capstone — Phase 1 refinement: one decoder trained on 2×T4 (DDP)

One model (your `arch1` / `arch3` code, unchanged) trained data-parallel on both T4s, then generation with every
decoding strategy in `DECODING` → `submission_<TAG>_<strategy>.zip`.

**Inputs (Add Input):**
1. the prebuilt data (`notebook08c3659751` output, `pretrain_data_bpe/`) **or** your A/B run's output — any folder
   with `legal_bpe.json` + `train.bin` + `val.bin` is found automatically;
2. only for a warm start (`INIT_FROM`): the A/B run's output, which holds `run_B/best.pt`.
   A warm start needs B's exact tokenizer: the config cell picks the data folder whose `legal_bpe.json`
   matches B's, and stops if none does.

**Runs:**
* default = **training only**, warm start from B (`INIT_FROM = "auto"`), probe first then straight into training
  (`RUN_PROBE = True`, `PROBE_ONLY = False`); fill the `None` hyperparameters → Save & Run All (commit), GPU T4 x2;
* afterwards download `run_<TAG>/best.pt` and generate locally (`genlocal/gen_local.py`).

## Stage 0 — session clock, login, configuration

In [ ]:
# session clock + GPU check (run first: the training deadline is measured from here)
import os, sys, json, time, math, glob, shutil, random, hashlib, subprocess
SESSION_START = time.time()

import numpy as np
import torch

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

N_GPUS = torch.cuda.device_count()
print("torch", torch.__version__, "| CPUs:", os.cpu_count(), "| GPUs:", N_GPUS,
      [torch.cuda.get_device_name(i) for i in range(N_GPUS)])

In [ ]:
# Hugging Face login from Kaggle Secrets (never paste the token into the notebook)
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

In [ ]:
# configuration
SEED = 42

# ---------------- session budget (plumbing) ----------------
SESSION_HOURS = 12.0        # Kaggle hard limit
SAFETY_MIN = 20             # stop everything this long before the limit
GEN_RESERVE_MIN = 10        # training-only run: final eval + checkpoint save. Set 45 if DECODING is not empty

# ---------------- inputs ----------------
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
B_TOKENIZER_SHA256 = "38f51f147c54fa95b86f6d14fa521b6edc7cbe2655b66bf1325c5f13e60dd030"  # legal_bpe.json of the A/B run
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
VOCAB_SIZE = 32_000


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# =====================================================================================
# YOUR CHOICES (course policy: model design + hyperparameters are yours). Fill every None.
# =====================================================================================
TAG = "B2"                           # name of this run: run_<TAG>/, submission_<TAG>_<strategy>.zip
INIT_FROM = "auto"                   # None = train from scratch | "auto" = warm start (weights only, fresh optimiser and
                                     # schedule) from an attached run_B/best.pt | or an explicit path to a best.pt
RESUME = None                        # path to a last.pt of THIS run (continue an interrupted run, same schedule)

MODEL = {                            # your B architecture as trained before (arch3 = GQA + RoPE); change it if you want
    "module": "arch3",
    "model": dict(vocab_size=VOCAB_SIZE, d_model=768, n_layers=18, n_heads=12, n_kv_heads=4, d_ff=3456,
                  max_seq_len=896, dropout=0.1, pos_emb_type="rope", norm_first=True, attention_scale=0.125),
    "init_std": 0.02,
}
TRAIN = dict(
    batch_tokens=16384,              # GLOBAL batch in tokens, split over the GPUs: batch_tokens // max_seq_len sequences
                                     # must be divisible by 2 (B's run: 8192 -> 9 seqs, one T4, micro-batch 9 fitted)
    lr=3e-4, min_lr=3e-5, warmup_steps=1000, schedule="cosine",   # "cosine" | "linear" | "constant"
    weight_decay=0.1, betas=(0.9, 0.95), eps=1e-8, grad_clip=1.0,  # B's previous values
    max_tokens=None,                 # optional token cap (global tokens); None = the schedule runs to the time deadline
    # plumbing
    eval_every_min=30, eval_windows=360, log_every=50, compile=True, fast_attention=True, seed=SEED,
)
GEN = dict(                          # settings shared by every decoding strategy
    checkpoint="best",               # "best" (lowest val loss) or "last"
    target_words=128, max_new_tokens=256, batch_size=32,
    n_val_prompts=100,               # held-out SC prompts -> val_generations_<name>.json for score_local.py
    seed=SEED,
)
DECODING = {}                        # {} = TRAINING ONLY (generate locally from best.pt with genlocal/gen_local.py).
                                     # To generate on Kaggle instead, add strategies and set GEN_RESERVE_MIN = 45, e.g.
                                     # "greedy": dict(temperature=0.0, top_k=None, top_p=1.0, repetition_penalty=1.0),
                                     # optional keys: no_repeat_ngram (n > 1, token n-grams), ngram_scope ("output" | "all")
if DECODING and GEN_RESERVE_MIN < 30:
    print(f"WARNING: {len(DECODING)} decoding strategies but GEN_RESERVE_MIN = {GEN_RESERVE_MIN}: generation may be skipped")
print("Mode:", "train + generate on Kaggle" if DECODING else "TRAINING ONLY (no generation on Kaggle)")

# ---------------- data folder (plumbing) ----------------
_cands = sorted({os.path.dirname(h) for h in glob.glob("/kaggle/input/**/legal_bpe.json", recursive=True)
                 if all(os.path.exists(os.path.join(os.path.dirname(h), f)) for f in ("train.bin", "val.bin"))})
assert _cands, "no data attached: add an input with legal_bpe.json + train.bin + val.bin (prebuilt data or the A/B output)"
_hash = {d: _sha256(os.path.join(d, "legal_bpe.json")) for d in _cands}
for d in _cands:
    print(f"data candidate: {d}  tokenizer sha256 {_hash[d][:12]}{'  (= B tokenizer)' if _hash[d] == B_TOKENIZER_SHA256 else ''}")
_match = [d for d in _cands if _hash[d] == B_TOKENIZER_SHA256]
if INIT_FROM:
    assert _match, "INIT_FROM needs B's tokenizer, but no attached data folder has it: attach the A/B run's output"
DATA_DIR = (_match or _cands)[0]
TOKENIZER_PATH = os.path.join(DATA_DIR, "legal_bpe.json")
print("Data dir:", DATA_DIR)

if INIT_FROM == "auto":
    _ck = sorted(glob.glob("/kaggle/input/**/run_B/best.pt", recursive=True))
    assert _ck, "INIT_FROM = 'auto' but no run_B/best.pt is attached (add the A/B run's output as an input)"
    INIT_FROM = _ck[0]
if INIT_FROM:
    assert os.path.exists(INIT_FROM), INIT_FROM
    print("Warm start from:", INIT_FROM)

os.makedirs(OUTPUT_DIR, exist_ok=True)


def missing(d, prefix=""):
    out = []
    for k, v in d.items():
        if isinstance(v, dict):
            out += missing(v, f"{prefix}{k}.")
        elif v is None and k not in ("attention_scale", "init_std", "max_tokens", "top_k"):
            out.append(prefix + k)
    return out

todo = missing(MODEL, "MODEL.") + missing(TRAIN, "TRAIN.")
assert not todo, "Fill these in the configuration cell before running: " + ", ".join(todo)
print(f"Configuration loaded. Run {TAG}: {MODEL['module']}, {N_GPUS} GPU(s)")

## Stage 1 — code files (written into `/kaggle/working`)

In [ ]:
%%writefile a1_tokenizer.py
# Legal byte-level BPE (your A1 tokenizer, optimised)
# tokenizer.py
#
# Byte-level BPE with an end-of-word marker (A1 design), adapted for the CS781 legal corpus.
#
# Changes vs. the A1 version (original kept in Assignment3/a1_tokenizer.py):
#   * Tokens are integer ids backed by raw bytes, not strings. Previously the byte "<", "/",
#     "w", ">" could merge into a token identical to the "</w>" marker (or to "<pad>"), and
#     decode() would then insert spurious spaces. Ids make that collision impossible.
#   * Legal-text pre-tokenisation inside each whitespace word: letter runs, digit groups of
#     at most 3, and punctuation runs are merged separately ("s.302(1)," -> s . 302 ( 1 ) ,),
#     so "court", "court," and "court." share one vocabulary entry instead of three, and
#     years / section numbers / citations don't flood the vocabulary. Only the last piece of
#     each whitespace word carries </w>, so decode(encode(x)) == " ".join(x.split()).
#   * Incremental BPE training (pair counts + inverted index + lazy max-heap): each merge
#     only touches the words that contain the merged pair, instead of recounting every pair
#     in the corpus for every merge. 32k merges on a legal sample take minutes, not days.
#   * Encoding uses merge ranks (GPT-2 style, same result as applying merges in order) and a
#     bounded per-word cache (the old lru_cache on a method also cached `self` and never freed).
#   * Extra reserved special tokens for later phases (cls/sep/mask, span-corruption sentinels).
#   * encode_many() encodes many documents in parallel worker processes.
#
# Assignment 3 contract is unchanged: train_tokenizer(train_text, vocab_size) returns an object
# with encode(text) -> list[int], decode(ids) -> str and a vocab_size attribute.

import re
import os
import heapq
import string
import json
from collections import Counter, defaultdict


# ============================================================
# SPECIAL TOKENS
# ============================================================

N_SENTINELS = 25

SPECIALS = (
    ["<pad>", "<unk>", "<s>", "</s>", "<cls>", "<sep>", "<mask>"]
    + [f"<extra_id_{i}>" for i in range(N_SENTINELS)]
)   # 32 specials: ids 0..31

MIN_COUNT = 2

# Pieces inside one whitespace word: letter runs | digit groups (<= 3) | punctuation runs | underscores.
# The alternatives cover every non-space character, so "".join(pieces) == word always holds.
PRETOKENIZE = re.compile(r"[^\W\d_]+|\d{1,3}|[^\w\s]+|_+")

MAX_CACHE_WORDS = 2_000_000


# ============================================================
# TEXT CLEANING (A1 helpers, unchanged; not used by the BPE)
# ============================================================

def check_valid_en(word):
    is_en = False
    is_hi = False
    has_digit = False

    for ch in word:
        code = ord(ch)

        # English alphabet
        if 0x0041 <= code <= 0x005A or 0x0061 <= code <= 0x007A:
            is_en = True

        # Arabic numerals
        elif 0x0030 <= code <= 0x0039:
            has_digit = True

        # Devanagari numerals
        elif 0x0966 <= code <= 0x096F:
            has_digit = True

        # Devanagari
        elif 0x0900 <= code <= 0x097F:
            is_hi = True

    if has_digit:
        return False

    return is_en and not is_hi


def clean_text_and_tokenize_en(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    return [word for word in tokens if check_valid_en(word)]


# ============================================================
# BYTE-LEVEL BPE
# ============================================================

def split_word(word):
    """Whitespace word -> list of (piece, is_last_piece)."""
    pieces = PRETOKENIZE.findall(word) or [word]
    last = len(pieces) - 1
    return [(p, i == last) for i, p in enumerate(pieces)]


class ByteBPE:

    def __init__(self, vocab_size=16000):

        self.vocab_size = vocab_size

        # Id layout: specials | 256 bytes | </w> | one id per learned merge
        self.n_special = len(SPECIALS)
        self.byte_offset = self.n_special
        self.eow_id = self.byte_offset + 256

        self.merges = []            # list of (left_id, right_id); merge k creates id eow_id + 1 + k
        self._reset_tables()

    # --------------------------------------------------------
    # VOCAB TABLES
    # --------------------------------------------------------

    def _reset_tables(self):
        self.token_bytes = [b""] * self.n_special + [bytes([b]) for b in range(256)] + [b""]
        self.token_eow = [False] * (self.n_special + 256) + [True]
        for left, right in self.merges:
            self.token_bytes.append(self.token_bytes[left] + self.token_bytes[right])
            self.token_eow.append(self.token_eow[right])
        self.ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.merge_to_id = {pair: self.eow_id + 1 + rank for rank, pair in enumerate(self.merges)}
        # decode table: bytes of each token, plus a space if it ends a word; specials decode to nothing
        self._decode_table = [
            b"" if i < self.n_special else tb + (b" " if eow else b"")
            for i, (tb, eow) in enumerate(zip(self.token_bytes, self.token_eow))
        ]
        self.itos = list(SPECIALS) + [self._display(i) for i in range(self.n_special, len(self.token_bytes))]
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.vocab = self.itos
        self._cache = {}

    def _display(self, idx):
        """Human-readable token string (for inspection only; never used for encoding)."""
        text = self.token_bytes[idx].decode("utf-8", errors="backslashreplace")
        return text + ("</w>" if self.token_eow[idx] else "")

    @property
    def special_ids(self):
        return {tok: i for i, tok in enumerate(SPECIALS)}

    @property
    def eot_id(self):
        """Document separator used in the pretraining stream."""
        return SPECIALS.index("</s>")

    def get_vocab_size(self):
        return len(self.token_bytes)

    # --------------------------------------------------------
    # PIECE -> INITIAL SYMBOLS
    # --------------------------------------------------------

    def _piece_symbols(self, piece, is_last):
        symbols = [self.byte_offset + b for b in piece.encode("utf-8")]
        if is_last:
            symbols.append(self.eow_id)
        return symbols

    # --------------------------------------------------------
    # TRAIN BPE
    # --------------------------------------------------------

    @staticmethod
    def count_words(texts):
        """Whitespace-word frequencies of an iterable of texts."""
        counts = Counter()
        for text in texts:
            counts.update(str(text).split())
        return counts

    def fit(self, tokenized_corpus=None, target_vocab_size=None, min_count=MIN_COUNT,
            word_counts=None, min_word_freq=1, verbose=True):
        """Learn merges.

        tokenized_corpus: iterable of lists of whitespace words (A1/A3 interface), or
        word_counts:      a precomputed {whitespace word: frequency} mapping (faster for big corpora).
        min_word_freq:    ignore whitespace words rarer than this while learning merges (speed knob).
        """
        target = self.vocab_size if target_vocab_size is None else target_vocab_size
        self.vocab_size = target

        if word_counts is None:
            word_counts = Counter()
            for sentence_tokens in tokenized_corpus:
                word_counts.update(sentence_tokens)

        # whitespace words -> (piece, is_last) frequencies
        piece_freq = Counter()
        for word, count in word_counts.items():
            if count < min_word_freq:
                continue
            for piece in split_word(word):
                piece_freq[piece] += count

        words = [self._piece_symbols(p, last) for p, last in piece_freq]
        freqs = list(piece_freq.values())
        del piece_freq

        if verbose:
            print("=" * 60)
            print("BUILDING BYTE-LEVEL BPE")
            print("=" * 60)
            print(f"Unique pre-token pieces: {len(words):,}")
            print(f"Initial vocabulary size: {self.eow_id + 1:,}")
            print(f"Target vocabulary size: {target:,}")

        # pair -> total count, pair -> set of word indices containing it
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for wi, (symbols, f) in enumerate(zip(words, freqs)):
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += f
                where[pair].add(wi)

        # max-heap on count with deterministic tie-break on ids; stale entries are skipped lazily
        heap = [(-c, pair) for pair, c in pair_counts.items()]
        heapq.heapify(heap)

        self.merges = []
        next_id = self.eow_id + 1

        while next_id < target and heap:
            neg, pair = heapq.heappop(heap)
            current = pair_counts.get(pair, 0)
            if current != -neg:
                if current > 0:
                    heapq.heappush(heap, (-current, pair))
                continue
            if current < min_count:
                if verbose:
                    print(f"\nStopping: best pair frequency = {current}")
                break

            left, right = pair
            new_id = next_id
            next_id += 1
            self.merges.append(pair)
            touched = Counter()

            for wi in list(where.pop(pair, ())):
                symbols, f = words[wi], freqs[wi]
                # remove this word's old pairs
                for p in zip(symbols, symbols[1:]):
                    pair_counts[p] -= f
                    touched[p] += 0
                    s = where.get(p)
                    if s is not None:
                        s.discard(wi)
                # merge left-to-right
                merged, i, n = [], 0, len(symbols)
                while i < n:
                    if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                        merged.append(new_id)
                        i += 2
                    else:
                        merged.append(symbols[i])
                        i += 1
                words[wi] = merged
                # add the new pairs
                for p in zip(merged, merged[1:]):
                    pair_counts[p] += f
                    touched[p] += 0
                    where[p].add(wi)

            pair_counts.pop(pair, None)
            for p in touched:
                c = pair_counts.get(p, 0)
                if c <= 0:
                    pair_counts.pop(p, None)
                    where.pop(p, None)
                elif new_id in p:
                    heapq.heappush(heap, (-c, p))   # only pairs with the new symbol can grow

            if verbose and len(self.merges) % 1000 == 0:
                print(f"Merges: {len(self.merges):,} | Vocab: {next_id:,} | Latest pair count: {current:,}")

        self._reset_tables()

        if verbose:
            print("\n" + "=" * 60)
            print("BPE TRAINING COMPLETE")
            print("=" * 60)
            print(f"Final vocabulary size: {self.get_vocab_size():,}")
            print(f"Total merges learned: {len(self.merges):,}")

        return self

    # --------------------------------------------------------
    # ENCODE
    # --------------------------------------------------------

    def _bpe(self, symbols):
        """Repeatedly merge the lowest-rank adjacent pair (== applying merges in learned order)."""
        ranks = self.ranks
        while len(symbols) > 1:
            best, best_rank = None, None
            for pair in zip(symbols, symbols[1:]):
                r = ranks.get(pair)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = pair, r
            if best is None:
                break
            new_id = self.merge_to_id[best]
            left, right = best
            merged, i, n = [], 0, len(symbols)
            while i < n:
                if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(symbols[i])
                    i += 1
            symbols = merged
        return symbols

    def encode_word(self, word):
        ids = self._cache.get(word)
        if ids is None:
            ids = []
            for piece, is_last in split_word(word):
                ids.extend(self._bpe(self._piece_symbols(piece, is_last)))
            ids = tuple(ids)
            if len(self._cache) >= MAX_CACHE_WORDS:
                self._cache.clear()
            self._cache[word] = ids
        return ids

    def encode_tokens(self, sentence_tokens):
        ids = []
        for word in sentence_tokens:
            ids.extend(self.encode_word(word))
        return ids

    def encode(self, text):
        """text -> list of integer ids (whitespace is normalised to single spaces)."""
        cache = self._cache
        ids = []
        for word in str(text).split():
            w = cache.get(word)
            if w is None:
                w = self.encode_word(word)
            ids.extend(w)
        return ids

    def pool(self, workers=None):
        """Process pool whose workers each hold a copy of this tokenizer (and keep their word cache).
        Use with the module-level worker functions, e.g. pool.imap(encode_array_worker, texts)."""
        import multiprocessing as mp
        workers = workers or os.cpu_count() or 1
        return mp.get_context("fork").Pool(workers, initializer=_worker_init, initargs=(self,))

    def encode_many(self, texts, workers=None, chunksize=64):
        """Encode many documents in parallel processes; yields lists of ids in input order."""
        with self.pool(workers) as pool:
            yield from pool.imap(_worker_encode, texts, chunksize=chunksize)

    # --------------------------------------------------------
    # DECODE
    # --------------------------------------------------------

    def decode(self, ids):
        table = self._decode_table
        n = len(table)
        out = b"".join(table[int(i)] if 0 <= int(i) < n else b"" for i in ids)
        return out.decode("utf-8", errors="replace").strip()

    def ends_word(self, idx):
        """True if token idx closes a whitespace word (used to count generated words)."""
        return self.token_eow[idx]

    # --------------------------------------------------------
    # SAVE / LOAD
    # --------------------------------------------------------

    def to_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "format": "legal-bytebpe-v2",
                    "specials": SPECIALS,
                    "merges": self.merges,
                    "vocab_size": self.get_vocab_size(),
                    "itos": self.itos,          # readable, informational only
                },
                f,
                ensure_ascii=False,
            )
        print(f"Saved tokenizer to: {path}")

    @classmethod
    def from_json(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if data.get("format") != "legal-bytebpe-v2":
            raise ValueError(f"{path} was not written by this tokenizer version")
        if data["specials"] != SPECIALS:
            raise ValueError("special tokens differ from this tokenizer version")
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = [tuple(pair) for pair in data["merges"]]
        tokenizer._reset_tables()
        return tokenizer


# worker-process helpers for encode_many (module level so they pickle)
_WORKER_TOKENIZER = None


def _worker_init(tokenizer):
    global _WORKER_TOKENIZER
    _WORKER_TOKENIZER = tokenizer


def _worker_encode(text):
    return _WORKER_TOKENIZER.encode(text)


def encode_array_worker(text):
    """(uint16 id array, whitespace word count) for one document; for tokenizer.pool().imap."""
    import numpy as np
    return np.asarray(_WORKER_TOKENIZER.encode(text), dtype=np.uint16), text.count(" ") + 1


# ============================================================
# ASSIGNMENT 3 TRAINING FUNCTION
# ============================================================

def train_tokenizer(train_text: str, vocab_size: int):
    text = str(train_text)
    print(f"Training tokenizer on {len(text):,} characters")
    tokenizer = ByteBPE(vocab_size=vocab_size)
    tokenizer.fit(
        word_counts=ByteBPE.count_words(text.splitlines()),
        target_vocab_size=vocab_size,
        min_count=MIN_COUNT,
    )
    return tokenizer


# ============================================================
# SIMPLE SMOKE TEST
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("TOKENIZER SMOKE TEST")
    print("=" * 60)

    train_text = """
    the court has jurisdiction over this matter under Section 302(1) of the Code.
    the court shall consider the evidence, vide AIR 1966 SC 1882, dated 18.12.1961.
    the evidence presented before the court </w> <pad> was 12,345 pages.
    """

    tokenizer = train_tokenizer(train_text, vocab_size=400)

    for test_text in [
        "the court has jurisdiction",
        "Section 302(1), AIR 1966 SC 1882 </w><pad> naïve — ₹5,00,000/- (u/s 138)",
    ]:
        ids = tokenizer.encode(test_text)
        decoded = tokenizer.decode(ids)
        print("\nInput:  ", test_text)
        print("Ids:    ", ids)
        print("Pieces: ", [tokenizer.itos[i] for i in ids])
        print("Decoded:", decoded)
        assert decoded == " ".join(test_text.split()), "round trip failed"

    print("\nVocabulary size:", tokenizer.get_vocab_size())

    tokenizer.to_json("bpe_smoke_test.json")
    loaded = ByteBPE.from_json("bpe_smoke_test.json")
    assert loaded.encode(test_text) == tokenizer.encode(test_text)
    os.remove("bpe_smoke_test.json")

    print("\n" + "=" * 60)
    print("SMOKE TEST PASSED")
    print("=" * 60)

In [ ]:
%%writefile arch1.py
# Architecture 1 — attention1.py + decoder1.py (your code)
# ======== attention1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Multi-head causal self-attention, implemented from raw tensor ops."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # Hook for Q1.4. None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        # TODO: define self.q_proj, self.k_proj, self.v_proj, self.out_proj as nn.Linear
        # (no bias). All should map d_model -> d_model.
        self.q_proj = nn.Linear(d_model,d_model,bias=False)
        self.k_proj = nn.Linear(d_model,d_model,bias=False)
        self.v_proj = nn.Linear(d_model,d_model,bias=False)
        self.out_proj = nn.Linear(d_model,d_model,bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # TODO: register a (max_seq_len, max_seq_len) lower-triangular boolean mask as a
        # buffer named `causal_mask` (True = allowed to attend, False = masked out), so
        # you don't rebuild it on every forward call.
        causal_mask = torch.tril(
            torch.ones(max_seq_len,max_seq_len,dtype=torch.bool)
        )
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        # TODO 1: project to q, k, v  (each (B, T, C))
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        # TODO 2: reshape each to (B, n_heads, T, d_head) — split C into (n_heads, d_head)
        #         and move n_heads next to the batch dim.
        q = torch.reshape(q,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        k = torch.reshape(k,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        v = torch.reshape(v,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        
        # TODO 3: compute attention scores = (q @ k^T) * self.attention_scale
        #         -> (B, n_heads, T, T). Do NOT hard-code 1/sqrt(d_head): Q1.4
        #         changes this hook for the scaling ablation.
        attention_scores = (q@k.transpose(2,3))*self.attention_scale
        # TODO 4: apply the causal mask (set disallowed positions to -inf before softmax)
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
        # TODO 5: softmax over the last dim, apply self.attn_dropout
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
        # TODO 6: weighted sum with v -> (B, n_heads, T, d_head)
        weighted_sum = attention_weights@v
        # TODO 7: reshape back to (B, T, C) and apply self.out_proj + self.resid_dropout
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM, built from scratch (no nn.Transformer*)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          # == n_heads for plain MHA; used in Question 2
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # or "rope", used in Question 3
    attention_scale: float = None      # Q1.4 hook; None -> 1/sqrt(d_head)
    norm_first: bool = False            # norm_first=True is for pre-norm else post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        # TODO: implement pre-norm when norm_first=True,
        # cfg.norm_first = true for pre norm
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # (RoPE is applied inside attention instead of here — see Question 3)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x

In [ ]:
%%writefile arch3.py
# Architecture 2 option — rope.py + attention3.py + decoder3.py (your code: GQA + RoPE)
# ======== rope.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated


# ======== attention3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile fastattn.py
# Plumbing: fused SDPA attention + KV cache + generation around your modules
"""Efficiency plumbing around the hand-written A3 models (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay
exactly as defined in arch1/arch2/arch3. `patch_model` swaps each attention module's forward
for one that computes the *same* function with fused kernels, and adds an optional KV cache
for generation. `check_equivalence` in the notebook verifies outputs match the original code.

* Training: F.scaled_dot_product_attention(is_causal=True). On T4 (sm_75) PyTorch cannot use
  the FlashAttention-2 kernel (needs sm_80+); it dispatches to the memory-efficient fused
  kernel, which gives the same benefit: no (T x T) score matrix materialised in HBM.
* Generation: per-layer KV cache (keys/values for MHA/GQA, the compressed latent for MLA),
  prefill once, then one token per step.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forward (drop-in for all three hand-written attention classes)
# ---------------------------------------------------------------------------

def _sdpa(q, k, v, causal, dropout_p, scale):
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal, dropout_p=dropout_p, scale=scale)


def _fast_attention_forward(self, x):
    B, T, C = x.shape
    cache = getattr(self, "_kv_cache", None)
    start = cache["pos"] if cache is not None else 0
    n_heads, d_head = self.n_heads, self.d_head
    dropout_p = self.attn_dropout.p if self.training else 0.0

    q = self.q_proj(x).view(B, T, n_heads, d_head).transpose(1, 2)

    if hasattr(self, "kv_down_proj"):                       # MLA: cache the shared latent
        latent = self.kv_down_proj(x)
        if cache is not None:
            cache["latent"][:, start:start + T] = latent
            latent = cache["latent"][:, :start + T]
        S = latent.shape[1]
        k = self.k_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
        v = self.v_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
    else:                                                   # MHA / GQA (+ optional RoPE)
        n_kv = getattr(self, "n_kv_heads", n_heads)
        k = self.k_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        if getattr(self, "use_rope", False):
            cos = self.rope_cos[start:start + T]
            sin = self.rope_sin[start:start + T]
            q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
            k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
        if cache is not None:
            cache["k"][:, :, start:start + T] = k
            cache["v"][:, :, start:start + T] = v
            k = cache["k"][:, :, :start + T]
            v = cache["v"][:, :, :start + T]
        n_rep = n_heads // n_kv
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)

    # prefill (start == 0) is causal; a single decode step attends to the whole cache
    causal = T > 1
    if causal and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    y = _sdpa(q, k, v, causal, dropout_p, self.attention_scale)
    y = y.transpose(1, 2).reshape(B, T, n_heads * d_head)
    return self.resid_dropout(self.out_proj(y))


def attention_modules(model):
    return [block.attn for block in model.blocks]


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    for attn in attention_modules(model):
        attn._kv_cache = None
        if getattr(attn, "use_rope", False):
            # the user's own apply_rope, from the module that defines this attention class
            attn._apply_rope = sys.modules[type(attn).__module__].apply_rope
        attn.forward = types.MethodType(_fast_attention_forward, attn)
    return model


# ---------------------------------------------------------------------------
# KV cache
# ---------------------------------------------------------------------------

@contextmanager
def kv_cache(model, batch_size, max_len, dtype):
    """Allocate a per-layer cache. Inside the context call `step(model, ids)` repeatedly."""
    dev = next(model.parameters()).device
    caches = []
    for attn in attention_modules(model):
        c = {"pos": 0}
        if hasattr(attn, "kv_down_proj"):
            c["latent"] = torch.empty(batch_size, max_len, attn.kv_latent_dim, device=dev, dtype=dtype)
        else:
            n_kv = getattr(attn, "n_kv_heads", attn.n_heads)
            shape = (batch_size, n_kv, max_len, attn.d_head)
            c["k"] = torch.empty(shape, device=dev, dtype=dtype)
            c["v"] = torch.empty(shape, device=dev, dtype=dtype)
        attn._kv_cache = c
        caches.append(c)
    full_pe = getattr(model, "pos_emb", None)
    model._fast_state = {"caches": caches, "full_pe": full_pe}
    try:
        yield
    finally:
        for attn in attention_modules(model):
            attn._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe
        del model._fast_state


def step(model, ids):
    """Run `ids` (B, T) through the model at the current cache position; returns logits (B, T, V)."""
    state = model._fast_state
    pos = state["caches"][0]["pos"]
    if state["full_pe"] is not None:
        # DecoderLM.forward adds self.pos_emb[:T]; shift the table so rows pos..pos+T are used
        model.pos_emb = state["full_pe"][pos:]
    logits = model(ids)
    for c in state["caches"]:
        c["pos"] += ids.shape[1]
    return logits


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p          # keep the smallest set with mass >= top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def ngram_ban(seq, n, vocab):
    """(B, V) bool: tokens that would complete an n-gram (in token ids) already present in `seq` (B, S).
    For each row, every earlier occurrence of the last n-1 tokens bans the token that followed it."""
    B, S = seq.shape
    if n < 2 or S < n:
        return torch.zeros(B, vocab, dtype=torch.bool, device=seq.device)
    grams = seq.unfold(1, n, 1)                                   # (B, S-n+1, n)
    hit = (grams[:, :, :-1] == seq[:, None, S - n + 1:]).all(-1)  # earlier (n-1)-gram == current suffix
    cnt = torch.zeros(B, vocab, device=seq.device).scatter_add_(1, grams[:, :, -1], hit.float())
    return cnt > 0


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens,
                   batch_size, temperature, top_k, top_p, repetition_penalty, device, amp_dtype,
                   log_every=10, no_repeat_ngram=0, ngram_scope="output"):
    """no_repeat_ngram = n > 1 forbids repeating any n-gram of TOKEN ids (HF no_repeat_ngram_size), counted over
    the generated text only (ngram_scope="output") or over prompt + generated text ("all")."""
    assert ngram_scope in ("output", "all"), ngram_scope
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    Prompts are bucketed by token length so a batch needs no padding: every prompt in a batch
    keeps its last L tokens, L = shortest prompt in the batch (capped so prompt + new tokens fit
    max_seq_len). Words are counted from the tokenizer's end-of-word flag, so we stop as soon as
    every row has produced target_words complete words.
    """
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    assert max_prompt > 0, "max_new_tokens must be smaller than max_seq_len"
    results = [None] * len(prompts)

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, B, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            history = ids
            for t in range(max_new_tokens):
                # a bare </w> right after a finished word never occurs in encoded text: forbid it
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = history if ngram_scope == "all" else (torch.stack(out, 1) if out else ids[:, :0])
                    ban = ngram_ban(seq, no_repeat_ngram, logits.size(1))
                    ban &= ~(ban | torch.isinf(logits)).all(-1, keepdim=True)  # never leave a row with no option
                    logits = logits.masked_fill(ban, -float("inf"))
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                # a word ends on an end-of-word token that closes something non-empty
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()):
                    break
                if repetition_penalty != 1.0 or (no_repeat_ngram and ngram_scope == "all"):
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(model, nxt[:, None])[:, -1]
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence(make_model, vocab_size, seq_len=48, rtol=1e-4):
    """Original forward vs patched forward vs KV-cached incremental decoding, fp32 on CPU.
    Returns max |difference| relative to the largest |logit| of the original model."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    with kv_cache(fast, 2, seq_len, torch.float32):
        pre = step(fast, idx[:, :seq_len // 2])
        steps = [step(fast, idx[:, t:t + 1]) for t in range(seq_len // 2, seq_len)]
    err_cache = (a - torch.cat([pre] + steps, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile train_ddp.py
# Plumbing: one decoder-only model trained data-parallel on every visible GPU, then multi-strategy generation
"""Train one hand-written A3 decoder (arch1 / arch3) on all GPUs with DistributedDataParallel, then write
prediction_<strategy>.json for every decoding strategy (written with Claude; declare it).

Usage:  torchrun --standalone --nproc_per_node=2 train_ddp.py run_B.json
        (also runs as a single process: python train_ddp.py run_B.json)

Same model / optimiser / schedule semantics as the single-GPU train_lm.py of the A/B run:
  * the global batch (batch_tokens) is split evenly over the GPUs; each GPU runs micro-batches of its share,
    gradients are averaged across GPUs (DDP), so one optimiser step still sees batch_tokens tokens;
  * the LR schedule is driven by wall-clock progress to `deadline_unix` (or max_tokens), computed on rank 0
    and broadcast every step, so all GPUs always use the same LR and stop at the same step;
  * eval, checkpoints, logging and generation happen on rank 0 only; checkpoints hold the plain (unwrapped)
    state_dict, loadable exactly like the A/B checkpoints (gen_local.py works unchanged).
Validation uses a fixed set of windows (eval_windows, default 360 = the A/B/C/D runs' 40 x 9), so val losses
are comparable with the earlier runs.

Probe mode (cfg["probe"]): same code path on synthetic tokens for a fixed number of minutes, then times
worst-case generation per decoding strategy instead of writing predictions.
"""
import os
import sys
import json
import math
import time
import queue
import random
import threading
import traceback
import gc
import importlib
from contextlib import nullcontext
from datetime import timedelta

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

import fastattn
from a1_tokenizer import ByteBPE

# ---------------------------------------------------------------------------
# distributed setup
# ---------------------------------------------------------------------------
WORLD = int(os.environ.get("WORLD_SIZE", "1"))
RANK = int(os.environ.get("RANK", "0"))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))
DIST = WORLD > 1
CUDA = torch.cuda.is_available()         # CPU path exists only for local smoke tests
if CUDA:
    torch.cuda.set_device(LOCAL_RANK)
device = torch.device(f"cuda:{LOCAL_RANK}" if CUDA else "cpu")
if DIST:
    dist.init_process_group("nccl" if CUDA else "gloo", timeout=timedelta(minutes=10))
MASTER = RANK == 0

cfg = json.load(open(sys.argv[1]))
TAG, OUT = cfg["tag"], cfg["out_dir"]
os.makedirs(OUT, exist_ok=True)
TR, GEN = cfg["train"], cfg["gen"]
DEADLINE = cfg["deadline_unix"]
SESSION_END = cfg["session_end_unix"]


def log(msg, force=False, **kv):
    """rank 0 logs everything; other ranks only what is passed with force=True (errors)."""
    if not (MASTER or force):
        return
    rec = {"t": round(time.time(), 1), "msg": msg, **kv}
    if not MASTER:
        rec["rank"] = RANK
    print(json.dumps(rec), flush=True)
    name = "log.jsonl" if MASTER else f"log_rank{RANK}.jsonl"
    with open(os.path.join(OUT, name), "a") as f:
        f.write(json.dumps(rec) + "\n")


seed = TR["seed"]
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)     # identical init on every rank
AMP = torch.float16                      # T4 has no bf16 tensor cores
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ---------------------------------------------------------------------------
# model
# ---------------------------------------------------------------------------
arch = importlib.import_module(cfg["module"])
mcfg = arch.LMConfig(**cfg["model"])
raw_model = arch.DecoderLM(mcfg)
if cfg.get("init_std"):
    std = cfg["init_std"]
    for m in raw_model.modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, std)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias)
n_params = sum(p.numel() for p in raw_model.parameters())
if TR.get("fast_attention", True):
    fastattn.patch_model(raw_model)
raw_model.to(device)
init_from = cfg.get("init_from")
if init_from and not (cfg.get("resume_from") and os.path.exists(cfg["resume_from"])):
    # warm start: weights only (fresh optimiser, fresh warmup + schedule); the architecture must match exactly
    ck = torch.load(init_from, map_location="cpu", weights_only=False)
    src = ck.get("config", {})
    assert src.get("module") == cfg["module"] and src.get("model") == cfg["model"], \
        f"init_from checkpoint architecture {src.get('module')} {src.get('model')} != this run's {cfg['module']} {cfg['model']}"
    raw_model.load_state_dict(ck["model"])
    log("init_from", path=init_from, src_step=ck.get("step"), src_tokens_M=round(ck.get("tokens", 0) / 1e6, 1),
        src_best_val=ck.get("best_val"))
    del ck
T = mcfg.max_seq_len
V = mcfg.vocab_size
assert GEN["max_new_tokens"] < T, "gen.max_new_tokens must be < max_seq_len"
log("model", tag=TAG, module=cfg["module"], params_M=round(n_params / 1e6, 2), seq_len=T, world_size=WORLD)

# ---------------------------------------------------------------------------
# data: random windows from the uint16 token stream, prefetched on a thread (different stream per rank)
# ---------------------------------------------------------------------------
train_mm = np.memmap(os.path.join(cfg["data_dir"], "train.bin"), dtype=np.uint16, mode="r")
val_mm = np.memmap(os.path.join(cfg["data_dir"], "val.bin"), dtype=np.uint16, mode="r")
assert int(train_mm[:10_000_000].max()) < V, "token id >= vocab_size: data/tokenizer mismatch"


def windows(mm, starts):
    arr = np.stack([mm[s:s + T + 1] for s in starts]).astype(np.int64)
    t = torch.from_numpy(arr)
    t = t.pin_memory() if CUDA else t
    return t[:, :-1], t[:, 1:]


class Prefetcher:
    def __init__(self, mb, seed):
        self.mb, self.rng = mb, np.random.default_rng(seed)
        self.q = queue.Queue(maxsize=6)
        threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        hi = len(train_mm) - T - 1
        while True:
            self.q.put(windows(train_mm, self.rng.integers(0, hi, self.mb)))

    def next(self):
        x, y = self.q.get()
        return x.to(device, non_blocking=True), y.to(device, non_blocking=True)


def loss_fn(model, x, y):
    with torch.autocast("cuda", dtype=AMP, enabled=CUDA):
        logits = model(x)
        return F.cross_entropy(logits.view(-1, logits.size(-1)), y.reshape(-1))


def all_min(v):
    if not DIST:
        return v
    t = torch.tensor([v], device=device, dtype=torch.long)
    dist.all_reduce(t, op=dist.ReduceOp.MIN)
    return int(t.item())


# ---------------------------------------------------------------------------
# batch: global batch split over the GPUs; micro-batch = largest divisor of the per-GPU share that fits
# ---------------------------------------------------------------------------
batch_seqs = max(1, TR["batch_tokens"] // T)
assert batch_seqs % WORLD == 0, (f"batch_tokens // max_seq_len = {batch_seqs} sequences must be divisible by the "
                                 f"{WORLD} GPUs (e.g. batch_tokens = {WORLD * math.ceil(batch_seqs / WORLD) * T})")
rank_seqs = batch_seqs // WORLD
free_mem = torch.cuda.mem_get_info()[0] + torch.cuda.memory_allocated() if CUDA else 0
opt_state_bytes = 2 * 4 * sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
ddp_bytes = 4 * n_params if DIST else 0                      # DDP gradient buckets
budget = free_mem - opt_state_bytes - ddp_bytes - 1.5e9
candidates = sorted([d for d in range(1, min(rank_seqs, TR.get("max_micro_batch", 64)) + 1)
                     if rank_seqs % d == 0], reverse=True)
mb, peak = None, 0
if not CUDA:
    mb = min(candidates, key=lambda d: abs(d - 2))
for cand in candidates if CUDA else []:
    try:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        x, y = windows(train_mm, np.arange(cand) * 997)
        raw_model.train()
        loss_fn(raw_model, x.to(device), y.to(device)).backward()
        raw_model.zero_grad(set_to_none=True)
        peak = torch.cuda.max_memory_allocated()
        if peak < budget:
            mb = cand
            break
    except torch.cuda.OutOfMemoryError:
        pass
    raw_model.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()
mb = all_min(mb if mb is not None else 0)                     # same micro-batch on every GPU
assert mb >= 1, "even micro-batch 1 does not fit; reduce max_seq_len or model size"
accum = rank_seqs // mb
log("batch", global_seqs=batch_seqs, gpus=WORLD, seqs_per_gpu=rank_seqs, micro_batch=mb, grad_accum=accum,
    tokens_per_step=batch_seqs * T, probe_peak_GB=round(peak / 1e9, 2))

ddp = DDP(raw_model, device_ids=[LOCAL_RANK] if CUDA else None, broadcast_buffers=False,
          gradient_as_bucket_view=True) if DIST else None
base = ddp if DIST else raw_model                              # what the training step calls
model = base
if TR.get("compile", False):
    t_compile = time.time()
    try:
        model = torch.compile(base)
        x, y = windows(train_mm, np.arange(mb) * 991)
        loss_fn(model, x.to(device), y.to(device)).backward()
        raw_model.zero_grad(set_to_none=True)
        log("torch.compile ok", seconds=round(time.time() - t_compile, 1))
    except Exception as e:                                     # fall back to eager
        model = base
        raw_model.zero_grad(set_to_none=True)
        log("torch.compile failed, using eager", error=repr(e)[:300])

# ---------------------------------------------------------------------------
# optimiser (weight decay on >=2-D weights only); one per rank, identical updates
# ---------------------------------------------------------------------------
decay = [p for p in raw_model.parameters() if p.requires_grad and p.dim() >= 2]
no_decay = [p for p in raw_model.parameters() if p.requires_grad and p.dim() < 2]
groups = [{"params": decay, "weight_decay": TR["weight_decay"]}, {"params": no_decay, "weight_decay": 0.0}]
try:
    opt = torch.optim.AdamW(groups, lr=TR["lr"], betas=tuple(TR["betas"]), eps=TR.get("eps", 1e-8), fused=CUDA)
except (RuntimeError, TypeError):
    opt = torch.optim.AdamW(groups, lr=TR["lr"], betas=tuple(TR["betas"]), eps=TR.get("eps", 1e-8))
scaler = torch.amp.GradScaler("cuda", enabled=CUDA)

step, tokens_seen, train_seconds_before, best_val = 0, 0, 0.0, float("inf")
warmup_end_progress = None
if cfg.get("resume_from") and os.path.exists(cfg["resume_from"]):
    ck = torch.load(cfg["resume_from"], map_location=device, weights_only=False)
    raw_model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    scaler.load_state_dict(ck["scaler"])
    step, tokens_seen = ck["step"], ck["tokens"]
    train_seconds_before, best_val = ck["train_seconds"], ck.get("best_val", float("inf"))
    warmup_end_progress = ck.get("warmup_end_progress")
    log("resumed", path=cfg["resume_from"], step=step, tokens=tokens_seen)

# ---------------------------------------------------------------------------
# schedule: warmup in steps, then decay over wall-clock progress (computed on rank 0, broadcast)
# ---------------------------------------------------------------------------
if cfg.get("probe"):                     # runtime probe: fixed training window from here
    DEADLINE = min(DEADLINE, time.time() + cfg["probe"]["train_minutes"] * 60)
t_start = time.time()
schedule_seconds = (cfg["schedule_hours"] * 3600 if cfg.get("schedule_hours")
                    else train_seconds_before + (DEADLINE - t_start))
max_tokens = TR.get("max_tokens")


def progress_now():
    p = (train_seconds_before + time.time() - t_start) / schedule_seconds
    if max_tokens:
        p = max(p, tokens_seen / max_tokens)
    return p


def lr_at(step, p):
    global warmup_end_progress
    lr, min_lr, warm = TR["lr"], TR["min_lr"], TR["warmup_steps"]
    if step < warm:
        return lr * (step + 1) / warm
    if warmup_end_progress is None:
        warmup_end_progress = min(p, 0.99)
    q = min(1.0, max(0.0, (p - warmup_end_progress) / (1 - warmup_end_progress)))
    kind = TR.get("schedule", "cosine")
    if kind == "cosine":
        return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * q))
    if kind == "linear":
        return lr - (lr - min_lr) * q
    return lr


def control():
    """rank 0 decides (lr, progress, stop); every rank receives the same values."""
    t = torch.zeros(3, dtype=torch.float64, device=device)
    if MASTER:
        p = progress_now()
        stop = time.time() >= DEADLINE or p >= 1.0
        t[0], t[1], t[2] = (0.0 if stop else lr_at(step, p)), p, float(stop)
    if DIST:
        dist.broadcast(t, src=0)
    return t[0].item(), t[1].item(), bool(t[2].item())


# ---------------------------------------------------------------------------
# eval + checkpoints (rank 0 only, no collectives)
# ---------------------------------------------------------------------------
val_starts = np.random.default_rng(1234).integers(0, len(val_mm) - T - 1, TR.get("eval_windows", 360))


@torch.no_grad()
def evaluate():
    raw_model.eval()
    tot, n = 0.0, 0
    for i in range(0, len(val_starts), mb):
        x, y = windows(val_mm, val_starts[i:i + mb])
        tot += loss_fn(raw_model, x.to(device), y.to(device)).item() * len(x)
        n += len(x)
    raw_model.train()
    return tot / n


def save(path, full):
    state = {"model": raw_model.state_dict(), "config": cfg, "step": step, "tokens": tokens_seen,
             "train_seconds": train_seconds_before + time.time() - t_start, "best_val": best_val,
             "warmup_end_progress": warmup_end_progress, "world_size": WORLD}
    if full:
        state.update(opt=opt.state_dict(), scaler=scaler.state_dict())
    torch.save(state, path + ".tmp")
    os.replace(path + ".tmp", path)


def do_eval(tag):
    global best_val
    t0 = time.time()
    vl = evaluate()
    t_eval = time.time() - t0
    improved = vl < best_val
    if improved:
        best_val = vl
        save(os.path.join(OUT, "best.pt"), full=False)
    save(os.path.join(OUT, "last.pt"), full=True)
    log(tag, step=step, tokens=tokens_seen, val_loss=round(vl, 4), val_ppl=round(math.exp(vl), 2),
        best=improved, hours=round((time.time() - t_start) / 3600, 2),
        eval_s=round(t_eval, 1), save_s=round(time.time() - t0 - t_eval, 1))


# ---------------------------------------------------------------------------
# train
# ---------------------------------------------------------------------------
pf = Prefetcher(mb, seed + 1000 * RANK + step)
raw_model.train()
last_eval = time.time()
last_log, tok_at_last_log = time.time(), tokens_seen
eval_every = TR["eval_every_min"] * 60
bad_logs = 0
log("training", deadline_in_h=round((DEADLINE - t_start) / 3600, 2), schedule_hours=round(schedule_seconds / 3600, 2))


def train_step(lr):
    for g in opt.param_groups:
        g["lr"] = lr
    loss_sum = torch.zeros((), device=device)
    for i in range(accum):
        x, y = pf.next()
        sync = (i == accum - 1) or not DIST
        with (nullcontext() if sync else ddp.no_sync()):       # all-reduce once per optimiser step
            loss = loss_fn(model, x, y) / accum
            scaler.scale(loss).backward()
        loss_sum += loss.detach()
    if TR.get("grad_clip"):
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), TR["grad_clip"]).item()
    else:
        gnorm = float("nan")
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    return loss_sum, gnorm


while True:
    try:
        lr, p, stop = control()
        if stop:
            break
        loss_sum, gnorm = train_step(lr)
    except Exception:
        # with several GPUs a failure on one rank cannot be recovered in step; stop cleanly instead
        opt.zero_grad(set_to_none=True)
        log("training error; stopping training (rank 0 then evaluates, saves and generates)", force=True,
            step=step, error=traceback.format_exc()[-1500:])
        break
    step += 1
    tokens_seen += batch_seqs * T

    if step % TR.get("log_every", 50) == 0:
        if DIST:
            dist.all_reduce(loss_sum, op=dist.ReduceOp.AVG if CUDA else dist.ReduceOp.SUM)
            if not CUDA:
                loss_sum /= WORLD
        loss_val = loss_sum.item()
        now = time.time()
        log("train", step=step, loss=round(loss_val, 4), lr=lr, grad_norm=round(gnorm, 3),
            tok_per_s=int((tokens_seen - tok_at_last_log) / (now - last_log)),
            tokens_M=round(tokens_seen / 1e6, 1), progress=round(p, 4),
            mem_GB=round(torch.cuda.max_memory_allocated() / 1e9, 2) if CUDA else 0.0)
        last_log, tok_at_last_log = now, tokens_seen
        bad_logs = 0 if math.isfinite(loss_val) else bad_logs + 1
        if bad_logs >= 3:
            log("loss non-finite for 3 log intervals; stopping training (generation uses the best checkpoint)",
                step=step)
            break                                              # every rank sees the same averaged loss
    if MASTER and time.time() - last_eval >= eval_every:
        do_eval("eval")
        last_eval = time.time()

del pf
if DIST and os.environ.get("DDP_CHECK"):                       # smoke tests: every rank must hold identical weights
    with torch.no_grad():
        s = torch.stack([p.double().sum() for p in raw_model.parameters()]).sum().view(1)
    sums = [torch.zeros_like(s) for _ in range(WORLD)]
    dist.all_gather(sums, s)
    log("ddp_check", param_sums=[round(x.item(), 6) for x in sums], identical=len({x.item() for x in sums}) == 1)
if not MASTER:                                                 # only rank 0 continues
    if DIST:
        try:
            dist.destroy_process_group()
        except Exception:
            pass
    sys.exit(0)

try:
    do_eval("final_eval")
except Exception:
    log("final eval failed", error=traceback.format_exc()[-1500:])
    save(os.path.join(OUT, "last.pt"), full=False)
log("training done", steps=step, tokens=tokens_seen, best_val=round(best_val, 4))
if DIST:
    try:
        dist.destroy_process_group()
    except Exception:
        pass

# ---------------------------------------------------------------------------
# generation (rank 0): one prediction file per decoding strategy (+ held-out val prompts for local scoring)
# ---------------------------------------------------------------------------
if not GEN.get("decoding"):                                    # training-only run: generate elsewhere from best.pt
    log("training only (no decoding strategies): best.pt / last.pt are the outputs")
    log("all done")
    sys.exit(0)

del opt
if CUDA:
    torch.cuda.empty_cache()
which = GEN.get("checkpoint", "best")
if not os.path.exists(os.path.join(OUT, f"{which}.pt")):
    log(f"{which}.pt missing (no finite val loss?); falling back to last.pt")
    which = "last"
ck = torch.load(os.path.join(OUT, f"{which}.pt"), map_location=device, weights_only=False)
raw_model.load_state_dict(ck["model"])
raw_model.eval()
if not TR.get("fast_attention", True):
    fastattn.patch_model(raw_model)          # KV cache needs the patched forward


class ProbeTokenizer:
    """Stand-in for the real tokenizer in probe mode: every prompt encodes to `prompt_len` random ids
    and no token ever ends a word, so each batch runs the full max_new_tokens (worst case)."""
    n_special, eow_id = 32, 0

    def __init__(self, vocab_size, prompt_len):
        self.token_eow = [False] * vocab_size
        self.token_bytes = [b"x"] * vocab_size
        self._ids = np.random.default_rng(0).integers(0, vocab_size, prompt_len).tolist()

    def encode(self, text):
        return self._ids

    def decode(self, ids):
        return ""


tokenizer = ProbeTokenizer(V, T - GEN["max_new_tokens"]) if cfg.get("probe") else ByteBPE.from_json(cfg["tokenizer_path"])
STRATEGIES = list(GEN["decoding"].items())         # [(name, {temperature, top_k, top_p, repetition_penalty}), ...]


def run_gen(prompts, dec):
    bs = GEN["batch_size"]
    while True:
        try:
            return fastattn.generate_words(
                raw_model, tokenizer, prompts, max_seq_len=T, target_words=GEN["target_words"],
                max_new_tokens=GEN["max_new_tokens"], batch_size=bs, temperature=dec["temperature"],
                top_k=dec.get("top_k"), top_p=dec.get("top_p"), repetition_penalty=dec.get("repetition_penalty", 1.0),
                device=device, amp_dtype=AMP if CUDA else torch.float32,
                no_repeat_ngram=dec.get("no_repeat_ngram", 0), ngram_scope=dec.get("ngram_scope", "output"))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            bs //= 2
            assert bs >= 1
            log("generation OOM, retrying", batch_size=bs)


def looping_share(texts):
    """share of predictions in which some 4-word sequence occurs 3+ times (a simple repetition indicator)."""
    n = 0
    for t in texts:
        w = t.split()
        grams = {}
        for i in range(len(w) - 3):
            g = tuple(w[i:i + 4])
            grams[g] = grams.get(g, 0) + 1
        n += bool(grams) and max(grams.values()) >= 3
    return round(n / max(1, len(texts)), 3)


if cfg.get("probe"):
    for name, dec in STRATEGIES:
        times = []
        for _ in range(2):                   # the first batch includes CUDA warm-up
            if CUDA:
                torch.cuda.synchronize()
            t0 = time.time()
            run_gen(["probe"] * GEN["batch_size"], dec)
            if CUDA:
                torch.cuda.synchronize()
            times.append(time.time() - t0)
        log("probe_generation", strategy=name, batch_size=GEN["batch_size"], prompt_tokens=T - GEN["max_new_tokens"],
            new_tokens=GEN["max_new_tokens"], first_batch_s=round(times[0], 2), batch_s=round(times[1], 2))
    log("all done")
    sys.exit(0)

test = json.load(open(cfg["test_path"], encoding="utf-8"))
ids = list(test.keys())
test_prompts = [test[k]["text"] for k in ids]
last_minutes = 0.0
for i, (name, dec) in enumerate(STRATEGIES):
    if i and SESSION_END - time.time() < 1.5 * last_minutes * 60 + 300:
        log("skipping decoding strategy: not enough session time left", strategy=name)
        continue
    torch.manual_seed(GEN.get("seed", 0))
    t0 = time.time()
    preds = run_gen(test_prompts, dec)
    last_minutes = (time.time() - t0) / 60
    prediction = {k: {"text": p} for k, p in zip(ids, preds)}
    paths = [os.path.join(OUT, f"prediction_{name}.json")] + ([os.path.join(OUT, "prediction.json")] if i == 0 else [])
    for path in paths:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(prediction, f, ensure_ascii=False, indent=2)
    short = sum(len(p.split()) < GEN["target_words"] for p in preds)
    log("prediction.json written", strategy=name, decoding=dec, n=len(preds), minutes=round(last_minutes, 1),
        under_target_words=short, looping_share=looping_share(preds), checkpoint=which)

vp_path = os.path.join(cfg["data_dir"], "val_prompts.json")
n_val = GEN.get("n_val_prompts", 0)
if n_val and os.path.exists(vp_path):
    vp = json.load(open(vp_path, encoding="utf-8"))
    keys = list(vp)[:n_val]
    for name, dec in STRATEGIES:
        if SESSION_END - time.time() < 600 + 0.25 * last_minutes * 60:
            log("skipping val generations: not enough session time left", strategy=name)
            break
        try:
            torch.manual_seed(GEN.get("seed", 0))
            vpreds = run_gen([vp[k]["text"] for k in keys], dec)
            with open(os.path.join(OUT, f"val_generations_{name}.json"), "w", encoding="utf-8") as f:
                json.dump({k: {"prediction": p, "reference": vp[k]["reference"]} for k, p in zip(keys, vpreds)},
                          f, ensure_ascii=False, indent=2)
            log("val_generations written", strategy=name, n=len(keys), looping_share=looping_share(vpreds))
        except Exception:
            log("val generation failed (prediction files are unaffected)", strategy=name,
                error=traceback.format_exc()[-1500:])
log("all done")

## Stage 2 — sanity checks (CPU)

In [ ]:
# sanity checks before spending GPU hours (~1 min)
import importlib
import fastattn, arch1, arch3
for m in (fastattn, arch1, arch3):
    importlib.reload(m)
from huggingface_hub import hf_hub_download

mod = importlib.import_module(MODEL["module"])
make = lambda: mod.DecoderLM(mod.LMConfig(**MODEL["model"]))
model = make()
n = sum(p.numel() for p in model.parameters())                  # tied embedding counted once
emb = model.tok_emb.weight.numel()
print(f"model {TAG} ({MODEL['module']}): {n / 1e6:.2f}M params  (embedding {emb / 1e6:.2f}M, non-embedding {(n - emb) / 1e6:.2f}M)")
assert 50e6 <= n <= 300e6, "Phase 1 requires 50M-300M parameters"
del model
T = MODEL["model"]["max_seq_len"]
assert GEN["max_new_tokens"] < T, f"GEN max_new_tokens ({GEN['max_new_tokens']}) must be < max_seq_len ({T})"
seqs = TRAIN["batch_tokens"] // T
gpus = max(1, N_GPUS)
assert seqs % gpus == 0, (f"batch_tokens // max_seq_len = {seqs} sequences cannot be split over {gpus} GPUs; "
                          f"nearest options: {(seqs // gpus) * gpus * T} or {(seqs // gpus + 1) * gpus * T} tokens")
print(f"   global batch {seqs} seqs x {T} = {seqs * T:,} tokens/step -> {seqs // gpus} seqs per GPU")
err_full, err_cache = fastattn.check_equivalence(make, MODEL["model"]["vocab_size"], seq_len=32)
print(f"   fused attention vs your forward: rel. max|diff| {err_full:.1e}; KV-cached decoding: {err_cache:.1e}  OK")
if INIT_FROM:
    _src = torch.load(INIT_FROM, map_location="cpu", weights_only=False)
    assert _src["config"]["module"] == MODEL["module"] and _src["config"]["model"] == json.loads(json.dumps(MODEL["model"])), \
        "INIT_FROM checkpoint has a different architecture than MODEL"
    print(f"   warm start: {INIT_FROM} (step {_src.get('step')}, {_src.get('tokens', 0) / 1e6:.0f}M tokens, "
          f"best val {_src.get('best_val'):.4f}); architecture matches")
    del _src

if DECODING:
    test_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN)
    test_data = json.load(open(test_path, encoding="utf-8"))
    print(f"   test prompts: {len(test_data)} | val_prompts.json present: {os.path.exists(os.path.join(DATA_DIR, 'val_prompts.json'))}")
else:
    test_path, test_data = None, {}
    print("   training only: test.json not needed")

## Stage 2b — runtime probe on 2×T4 (~25 min; optional, before committing to a 12 h run)

In [ ]:
# runtime probe: real train_ddp.py on synthetic tokens across all GPUs, then a projection for the full run
RUN_PROBE = True
PROBE_ONLY = False              # True: stop after the report. False: continue into the full run (probe time is spent)
PROBE_TRAIN_MIN = 10            # timed training window, after micro-batch search + torch.compile
PROBE_EVAL_EVERY_MIN = 3        # eval + checkpoint save a few times, to time them


class ProbeOnlyStop(Exception):
    pass


PROBE_DIR = os.path.join(WORK, "probe")
_median = lambda xs: float(np.median(xs)) if len(xs) else float("nan")


def _torchrun(cfg_path, logfile):
    env = dict(os.environ, OMP_NUM_THREADS="1", PYTHONUNBUFFERED="1")
    return subprocess.Popen([sys.executable, "-m", "torch.distributed.run", "--standalone",
                             f"--nproc_per_node={max(1, N_GPUS)}", "train_ddp.py", cfg_path],
                            cwd=WORK, env=env, stdout=open(logfile, "w"), stderr=subprocess.STDOUT)


if RUN_PROBE:
    t_probe = time.time()
    shutil.rmtree(PROBE_DIR, ignore_errors=True)
    probe_data = os.path.join(PROBE_DIR, "data")
    os.makedirs(probe_data)
    _rng = np.random.default_rng(SEED)
    _rng.integers(0, VOCAB_SIZE, 20_000_000, dtype=np.uint16).tofile(os.path.join(probe_data, "train.bin"))
    _rng.integers(0, VOCAB_SIZE, 2_000_000, dtype=np.uint16).tofile(os.path.join(probe_data, "val.bin"))
    pcfg = {"tag": TAG, **MODEL, "data_dir": probe_data, "out_dir": os.path.join(PROBE_DIR, f"run_{TAG}"),
            "tokenizer_path": None, "test_path": None, "init_from": None, "resume_from": None,
            "deadline_unix": SESSION_START + SESSION_HOURS * 3600 - SAFETY_MIN * 60,
            "session_end_unix": SESSION_START + SESSION_HOURS * 3600 - SAFETY_MIN * 60, "schedule_hours": None,
            "probe": {"train_minutes": PROBE_TRAIN_MIN},
            "train": dict(TRAIN, eval_every_min=PROBE_EVAL_EVERY_MIN, log_every=20, max_tokens=None),
            "gen": dict(GEN, decoding=DECODING)}
    ppath = os.path.join(WORK, "probe_cfg.json")
    json.dump(pcfg, open(ppath, "w"), indent=2)
    plog = os.path.join(PROBE_DIR, "probe.log")
    proc = _torchrun(ppath, plog)
    gpu_samples, last_print = [], time.time()
    _Q = "index,temperature.gpu,clocks.sm,power.draw,utilization.gpu,memory.used"
    while proc.poll() is None:
        try:
            out = subprocess.run(["nvidia-smi", f"--query-gpu={_Q}", "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True, timeout=20).stdout
            for line in out.strip().splitlines():
                f = [x.strip() for x in line.split(",")]
                try:
                    gpu_samples.append(dict(gpu=int(f[0]), temp=float(f[1]), sm=float(f[2]), power=float(f[3]),
                                            util=float(f[4]), mem_mb=float(f[5])))
                except ValueError:
                    pass
        except Exception:
            pass
        time.sleep(15)
        if time.time() - last_print > 120:
            last_print = time.time()
            lines = open(plog).read().strip().splitlines()
            print(f"{(time.time() - t_probe) / 60:5.1f} min | {lines[-1][:220] if lines else ''}", flush=True)
    probe_minutes = (time.time() - t_probe) / 60
    if proc.returncode:
        print(f"probe exited with code {proc.returncode}; last lines:")
        print("\n".join(open(plog).read().splitlines()[-40:]))

    recs = [json.loads(l) for l in open(os.path.join(PROBE_DIR, f"run_{TAG}", "log.jsonl"))] \
        if os.path.exists(os.path.join(PROBE_DIR, f"run_{TAG}", "log.jsonl")) else []
    by = lambda m: [r for r in recs if r["msg"] == m]
    tr = by("train")
    print("\n" + "=" * 100 + f"\nPROBE REPORT  (probe took {probe_minutes:.1f} min, {N_GPUS} GPUs, test prompts: {len(test_data)})\n" + "=" * 100)
    for r in recs:
        if any(w in r["msg"] for w in ("OOM", "error", "failed", "non-finite", "stopping")):
            print("WARNING:", json.dumps(r)[:400])
    assert tr, f"no training steps logged -> see {plog}"
    m, b = by("model")[0], by("batch")[-1]
    comp = by("torch.compile ok")
    setup_s = by("training")[0]["t"] - m["t"]
    tps = [r["tok_per_s"] for r in tr[1:]] or [r["tok_per_s"] for r in tr]
    third = max(1, len(tps) // 3)
    tps_use = min(_median(tps), _median(tps[-third:]))
    evals = by("eval") + by("final_eval")
    eval_s, save_s = _median([r["eval_s"] for r in evals]), _median([r["save_s"] for r in evals])
    gen = by("probe_generation")
    pre_h = (time.time() - SESSION_START) / 3600 + setup_s / 3600 if not PROBE_ONLY else (t_probe - SESSION_START + setup_s) / 3600
    budget_h = (SESSION_HOURS * 60 - SAFETY_MIN - GEN_RESERVE_MIN) / 60 - pre_h
    eval_frac = (eval_s + save_s) / (TRAIN["eval_every_min"] * 60)
    eff_tps = tps_use * (1 - eval_frac)
    tokens_budget = eff_tps * budget_h * 3600
    step_tokens = b["tokens_per_step"]
    n_test_b = math.ceil(len(test_data) / GEN["batch_size"])
    n_val_b = math.ceil(GEN.get("n_val_prompts", 0) / GEN["batch_size"])
    gen_min = sum(g["batch_s"] for g in gen) * (n_test_b + n_val_b) / 60
    print(f"model {TAG} ({m['module']}, {m['params_M']}M params, seq {m['seq_len']}) on {m['world_size']} GPU(s)")
    print(f"  batch      : {b['global_seqs']} seqs ({b['seqs_per_gpu']} per GPU, micro-batch {b['micro_batch']} x accum {b['grad_accum']}) "
          f"= {step_tokens:,} tokens/step | peak memory {max(r.get('mem_GB', 0) for r in tr):.1f} GB per GPU (of ~15 GB)")
    print(f"  setup      : {setup_s / 60:.1f} min (" + (f"compile {comp[0]['seconds']:.0f} s" if comp else "compile off/FAILED") + ")")
    print(f"  throughput : median {_median(tps):,.0f} tok/s for both GPUs together | first third {_median(tps[:third]):,.0f} -> "
          f"last third {_median(tps[-third:]):,.0f} | {tps_use / step_tokens:.2f} steps/s")
    print(f"  eval + save: {eval_s:.1f} s + {save_s:.1f} s every {TRAIN['eval_every_min']} min ({eval_frac:.1%} overhead)")
    for g in gen:
        print(f"  generation : {g['strategy']:>12s} {g['batch_s']:.1f} s per batch of {GEN['batch_size']} (worst case)")
    print(f"  12 h run   : training budget ~{budget_h:.2f} h -> ~{tokens_budget / 1e6:,.0f}M tokens at {eff_tps:,.0f} tok/s "
          f"({tokens_budget / step_tokens:,.0f} steps)")
    cap = TRAIN.get("max_tokens")
    if cap:
        t_cap = cap / eff_tps / 3600
        print(f"               max_tokens {cap / 1e6:,.0f}M " + (f"reached after ~{t_cap:.2f} h" if t_cap <= budget_h else
              f"NOT reached: deadline first at ~{tokens_budget / cap:.0%}; the schedule compresses to the deadline"))
    end_min = (eval_s + save_s) / 60 + gen_min
    print(f"  final eval + generation of all {len(gen)} strategies (test + val, worst case) ~{end_min:.1f} min vs "
          f"GEN_RESERVE_MIN {GEN_RESERVE_MIN} -> {'OK' if end_min < GEN_RESERVE_MIN else 'DOES NOT FIT: raise GEN_RESERVE_MIN'}")
    for gi in sorted({s["gpu"] for s in gpu_samples}):
        busy = [s for s in gpu_samples if s["gpu"] == gi and s["util"] >= 50]
        if busy:
            print(f"  GPU{gi}: util {_median([s['util'] for s in busy]):.0f}% | SM {_median([s['sm'] for s in busy]):.0f} MHz | "
                  f"temp {max(s['temp'] for s in busy):.0f} C | power {_median([s['power'] for s in busy]):.0f} W | "
                  f"mem {max(s['mem_mb'] for s in busy) / 1024:.1f} GB")
    for f in glob.glob(os.path.join(PROBE_DIR, "run_*", "*.pt")) + glob.glob(os.path.join(probe_data, "*.bin")):
        os.remove(f)
    if PROBE_ONLY:
        raise ProbeOnlyStop("Probe finished (PROBE_ONLY = True): read the report above. "
                            "Set RUN_PROBE = False for the full run.")

## Stage 4 — train on all GPUs (DDP), then generate every decoding strategy

In [ ]:
# full run: train_ddp.py under torchrun on every GPU; rank 0 evaluates, saves and generates
SESSION_END = SESSION_START + SESSION_HOURS * 3600 - SAFETY_MIN * 60
train_deadline = SESSION_END - GEN_RESERVE_MIN * 60
hours_left = (train_deadline - time.time()) / 3600
print(f"Training budget: {hours_left:.2f} h (session ends in {(SESSION_END - time.time()) / 3600:.2f} h)")
assert hours_left > 0.5, "not enough session time left for training"
assert N_GPUS >= 1, "no GPU visible: set Accelerator to 'GPU T4 x2'"

run_dir = os.path.join(OUTPUT_DIR, f"run_{TAG}")
cfg = {"tag": TAG, **MODEL, "data_dir": DATA_DIR, "out_dir": run_dir,
       "tokenizer_path": TOKENIZER_PATH, "test_path": test_path, "init_from": INIT_FROM, "resume_from": RESUME,
       "deadline_unix": train_deadline, "session_end_unix": SESSION_END, "schedule_hours": None,
       "train": TRAIN, "gen": dict(GEN, decoding=DECODING)}
cfg_path = os.path.join(WORK, f"run_{TAG}.json")
json.dump(cfg, open(cfg_path, "w"), indent=2)
train_log = os.path.join(OUTPUT_DIR, f"train_{TAG}.log")
proc = _torchrun(cfg_path, train_log)


def _recs():
    p = os.path.join(run_dir, "log.jsonl")
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []


while proc.poll() is None:
    time.sleep(600)
    recs = _recs()
    tr = [r for r in recs if r["msg"] == "train"]
    ev = [r for r in recs if r["msg"] in ("eval", "final_eval")]
    print(time.strftime("%H:%M:%S"), f"| {(time.time() - SESSION_START) / 3600:.2f} h into session", flush=True)
    if tr:
        k = tr[-12:]                                   # average of the last 12 logged steps (single steps are noisy)
        print(f"  train: step {tr[-1]['step']} | loss (avg of last {len(k)} logs) {np.mean([r['loss'] for r in k]):.3f} | "
              f"lr {tr[-1]['lr']:.2e} | grad_norm {tr[-1]['grad_norm']} | {tr[-1]['tok_per_s']:,} tok/s | "
              f"{tr[-1]['tokens_M']}M tokens | progress {tr[-1]['progress']:.3f} | {tr[-1]['mem_GB']} GB", flush=True)
    if ev:
        print(f"  val  : " + " -> ".join(f"{r['val_loss']:.3f}" for r in ev[-6:]) + f"  (best {min(r['val_loss'] for r in ev):.3f})", flush=True)
    other = [r for r in recs[-5:] if r["msg"] not in ("train", "eval", "final_eval")]
    for r in other:
        print("  " + json.dumps(r)[:260], flush=True)
print(f"run {TAG} exited with code {proc.returncode}")
if proc.returncode:
    print("\n".join(open(train_log).read().splitlines()[-40:]))

## Stage 5 — Codabench zips + curves

In [ ]:
# submissions + curves
import zipfile
from collections import Counter
import matplotlib.pyplot as plt

found = sorted(glob.glob(os.path.join(run_dir, "prediction_*.json"))) if DECODING else []   # one file per strategy
for pred in found:
    name = os.path.basename(pred)[len("prediction_"):-len(".json")]
    preds = json.load(open(pred, encoding="utf-8"))
    assert set(preds) == set(test_data), f"{name}: prediction ids differ from test ids"
    n_words = [len(v["text"].split()) for v in preds.values()]
    looping = np.mean([max(Counter(tuple(w[i:i + 4]) for i in range(len(w) - 3)).values(), default=0) >= 3
                       for w in (v["text"].split() for v in preds.values())])
    zname = f"submission_{TAG}_{name}.zip"
    with zipfile.ZipFile(os.path.join(OUTPUT_DIR, zname), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred, arcname="prediction.json")
    print(f"[{name:>12s}] {len(preds)} predictions, words min/mean {min(n_words)}/{np.mean(n_words):.1f}, "
          f"{looping:.0%} with a 4-word sequence repeated 3+ times -> {zname}")
if not DECODING:
    for f in ("best.pt", "last.pt"):
        p = os.path.join(run_dir, f)
        print(f"training only: {p} " + (f"({os.path.getsize(p) / 1e9:.2f} GB)" if os.path.exists(p) else "MISSING"))
    print("-> download best.pt and generate locally: genlocal/gen_local.py --ckpt B2=best.pt ...")
elif not found:
    print(f"no prediction files (see {train_log})")

recs = _recs()
tr = [r for r in recs if r["msg"] == "train"]
ev = [r for r in recs if r["msg"] in ("eval", "final_eval")]
if tr:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot([r["tokens_M"] for r in tr], [r["loss"] for r in tr], label=f"{TAG} train", alpha=0.6)
    axes[0].plot([r["tokens"] / 1e6 for r in ev], [r["val_loss"] for r in ev], "o-", label=f"{TAG} val")
    axes[1].plot([r["tokens_M"] for r in tr], [r["tok_per_s"] for r in tr], label=f"{TAG} ({N_GPUS} GPUs)")
    axes[0].set(xlabel="tokens (M)", ylabel="loss", title="train / val loss")
    axes[1].set(xlabel="tokens (M)", ylabel="tokens / s", title="throughput (all GPUs)")
    for ax in axes:
        ax.grid(True); ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
    plt.show()
print(sorted(os.listdir(OUTPUT_DIR)))